# YOLOv8 Fair Model Comparison — Bachelor Thesis

Trains **yolov8n**, **yolov8s**, and **yolov8m** under **strictly controlled, identical conditions** for a valid academic comparison.

### Why controlled conditions matter
Any difference in hyperparameters (learning rate, augmentation, epochs, optimizer, seed) other than model architecture will confound your results. This notebook ensures:
- Identical hyperparameters across all three models
- Same random seed → reproducible splits and weight initialization
- Same effective batch size via gradient accumulation (so n/s/m all see the same number of gradient steps per epoch)
- Same data augmentation pipeline
- Same optimizer and LR schedule
- Identical evaluation protocol

## 1 — Install dependencies

In [1]:
!pip install ultralytics roboflow --quiet
import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")

Ultralytics version: 8.4.50


## 2 — Verify GPU & log hardware (include this in your thesis)

In [2]:
import torch, platform, subprocess

print("=== Hardware & Software Environment ===")
print(f"PyTorch version   : {torch.__version__}")
print(f"CUDA version      : {torch.version.cuda}")
print(f"Python version    : {platform.python_version()}")
print(f"OS                : {platform.platform()}")

if torch.cuda.is_available():
    print(f"GPU               : {torch.cuda.get_device_name(0)}")
    print(f"VRAM              : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"cuDNN version     : {torch.backends.cudnn.version()}")
    # Enable deterministic ops for reproducibility
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print("cuDNN deterministic mode: ENABLED")
else:
    raise RuntimeError("No CUDA GPU found. Training on CPU is not viable.")

=== Hardware & Software Environment ===
PyTorch version   : 2.5.1+cu121
CUDA version      : 12.1
Python version    : 3.11.15
OS                : Windows-10-10.0.19045-SP0
GPU               : NVIDIA GeForce RTX 3060
VRAM              : 12.88 GB
cuDNN version     : 90100
cuDNN deterministic mode: ENABLED


## 3 — Download dataset from Roboflow

In [4]:
from roboflow import Roboflow
import os

rf = Roboflow(api_key="tgBGs8vzvXAfuZqaRpBe")   # <-- paste your key
project = rf.workspace("delfos").project("bundesliga-data-shootout-a411n")
version = project.version(9)
dataset = version.download("yolov8")

DATA_YAML = os.path.join(dataset.location, "data.yaml")
print(f"Dataset location  : {dataset.location}")
print(f"data.yaml path    : {DATA_YAML}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Bundesliga-Data-Shootout-9 in yolov8:: 100%|██████████| 2174/2174 [00:00<00:00, 2758.26it/s]

Dataset location  : e:\BsC_code\Bundesliga-Data-Shootout-9
data.yaml path    : e:\BsC_code\Bundesliga-Data-Shootout-9\data.yaml


## 4 — Inspect dataset (document in your thesis)

In [5]:
import yaml, glob

with open(DATA_YAML, 'r') as f:
    data_cfg = yaml.safe_load(f)

print("=== Dataset Summary ===")
print(f"Classes ({data_cfg['nc']})  : {data_cfg['names']}")

for split in ['train', 'valid', 'test']:
    split_path = os.path.join(dataset.location, split, 'images')
    n = len(glob.glob(split_path + '/*'))
    print(f"{split:10s} images: {n}")

=== Dataset Summary ===
Classes (2)  : ['ball', 'player']
train      images: 981
valid      images: 70
test       images: 30


## 5 — Define IDENTICAL training configuration

**Critical for thesis validity:**
- `seed=42` — same random state for all three runs
- `batch=8` with `nbs=64` — **nominal batch size trick**: YOLOv8 automatically scales the learning rate relative to `nbs`. Setting `nbs=64` and `batch=8` means 8 actual images per step but the LR is computed as if batch=64. This makes the *effective* training dynamics identical regardless of the physical batch size you're forced to use by VRAM. This is the academically correct way to handle VRAM constraints.
- All augmentation flags are explicit (not defaults) so they are locked and visible
- `amp=False` — disabling mixed precision makes floating-point ops identical; enable only if you run out of VRAM
- `optimizer='SGD'`, `lr0`, `momentum`, `weight_decay` all fixed

In [8]:
# ============================================================
#  SHARED CONFIG — every key here is IDENTICAL for n, s, m
# ============================================================
SEED   = 42
EPOCHS = 50
IMGSZ  = 640

# Physical batch per GPU pass — 8 fits all three models on RTX 3060 12 GB
# Increase to 16 if you have headroom (check with nvidia-smi during first epoch)
BATCH  = 8

# Nominal batch size — LR is scaled relative to this value.
# Keep at 64 (YOLOv8 default) so all three models use the same effective LR.
NBS    = 64

SHARED = dict(
    data          = DATA_YAML,
    epochs        = EPOCHS,
    imgsz         = IMGSZ,
    batch         = BATCH,
    nbs           = NBS,
    device        = 0,
    workers       = 4,
    seed          = SEED,
    deterministic = True,
    amp           = False,
    optimizer     = 'SGD',
    lr0           = 0.01,
    patience      = 20,
    cache         = 'ram',
    exist_ok      = True,
    plots         = True,
)

MODELS = [
    {"weights": "yolov8n.pt", "name": "yolov8n_thesis"},
    {"weights": "yolov8s.pt", "name": "yolov8s_thesis"},
    {"weights": "yolov8m.pt", "name": "yolov8m_thesis"},
]

print("Configuration locked. Shared hyperparameters:")
for k, v in SHARED.items():
    print(f"  {k:<20} = {v}")

Configuration locked. Shared hyperparameters:
  data                 = e:\BsC_code\Bundesliga-Data-Shootout-9\data.yaml
  epochs               = 50
  imgsz                = 640
  batch                = 8
  nbs                  = 64
  device               = 0
  workers              = 4
  seed                 = 42
  deterministic        = True
  amp                  = False
  optimizer            = SGD
  lr0                  = 0.01
  patience             = 20
  cache                = ram
  exist_ok             = True
  plots                = True


## 6 — Train all three models

In [9]:
from ultralytics import YOLO
import time, gc

results_log = []

for cfg in MODELS:
    print("\n" + "="*65)
    print(f"  MODEL: {cfg['weights']}  |  Run: {cfg['name']}")
    print("="*65)

    # Clear GPU memory between runs
    gc.collect()
    torch.cuda.empty_cache()

    model = YOLO(cfg["weights"])

    t0 = time.time()
    result = model.train(
        **SHARED,
        name = cfg["name"],
    )
    train_time = time.time() - t0

    rd = result.results_dict
    results_log.append({
        "Model"           : cfg["weights"].replace(".pt", ""),
        "Run"             : cfg["name"],
        "Precision"       : round(rd.get("metrics/precision(B)",  0), 4),
        "Recall"          : round(rd.get("metrics/recall(B)",     0), 4),
        "mAP@0.5"         : round(rd.get("metrics/mAP50(B)",      0), 4),
        "mAP@0.5:0.95"    : round(rd.get("metrics/mAP50-95(B)",   0), 4),
        "Train time (min)": round(train_time / 60, 1),
    })

    print(f"\n  Finished {cfg['weights']} in {train_time/60:.1f} min")
    print(f"  mAP@0.5 = {results_log[-1]['mAP@0.5']}  |  mAP@0.5:0.95 = {results_log[-1]['mAP@0.5:0.95']}")

print("\n\nAll models trained.")


  MODEL: yolov8n.pt  |  Run: yolov8n_thesis
New https://pypi.org/project/ultralytics/8.4.53 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.50  Python-3.11.15 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060, 12287MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=e:\BsC_code\Bundesliga-Data-Shootout-9\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train,

## 7 — Results table (copy into your thesis)

In [11]:
!pip install pandas --quiet

In [12]:
import pandas as pd

df = pd.DataFrame(results_log)
print("=== Thesis Comparison Table ===")
print(df.drop(columns=["Run"]).to_string(index=False))
df.to_csv("thesis_results.csv", index=False)
print("\nSaved to thesis_results.csv")

=== Thesis Comparison Table ===
  Model  Precision  Recall  mAP@0.5  mAP@0.5:0.95  Train time (min)
yolov8n     0.6273  0.5225   0.5015        0.1994              12.8
yolov8s     0.7399  0.6224   0.5985        0.2181              43.6
yolov8m     0.6560  0.5735   0.5532        0.2272              57.4

Saved to thesis_results.csv


## 8 — Per-class validation for each model

In [13]:
from ultralytics import YOLO
import pandas as pd

all_class_results = []

for cfg in MODELS:
    best = f"runs/detect/{cfg['name']}/weights/best.pt"
    model_name = cfg['weights'].replace('.pt', '')
    print(f"\nValidating {model_name} ({best})")

    model = YOLO(best)
    metrics = model.val(
        data    = DATA_YAML,
        imgsz   = IMGSZ,
        batch   = BATCH,
        device  = 0,
        split   = 'test',    # evaluate on held-out test set
        verbose = True,
    )

    # Overall
    print(f"  Overall mAP@0.5     : {metrics.box.map50:.4f}")
    print(f"  Overall mAP@0.5:0.95: {metrics.box.map:.4f}")

    # Per-class
    with open(DATA_YAML) as f:
        class_names = yaml.safe_load(f)["names"]

    for i, cls in enumerate(class_names):
        all_class_results.append({
            "Model"       : model_name,
            "Class"       : cls,
            "Precision"   : round(float(metrics.box.p[i]),  4),
            "Recall"      : round(float(metrics.box.r[i]),  4),
            "mAP@0.5"     : round(float(metrics.box.ap50[i]), 4),
            "mAP@0.5:0.95": round(float(metrics.box.ap[i]),   4),
        })

df_cls = pd.DataFrame(all_class_results)
print("\n=== Per-class Results ===")
print(df_cls.to_string(index=False))
df_cls.to_csv("thesis_per_class_results.csv", index=False)
print("\nSaved to thesis_per_class_results.csv")


Validating yolov8n (runs/detect/yolov8n_thesis/weights/best.pt)
Ultralytics 8.4.50  Python-3.11.15 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060, 12287MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 10.51.5 MB/s, size: 53.1 KB)
val: Scanning E:\BsC_code\Bundesliga-Data-Shootout-9\test\labels... 30 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 30/30 163.9it/s 0.2ss
val: New cache created: E:\BsC_code\Bundesliga-Data-Shootout-9\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.4s/it 5.8s0.7s4s
                   all         30        650      0.539      0.485      0.501       0.19
                  ball         28         32      0.222     0.0625     0.0793     0.0111
                player         30        618      0.857      0.908      0.923      0.368
Speed: 12.0ms preprocess, 34.7ms inference, 0.0ms loss, 7.

## 9 — Model size & parameter count

In [14]:
import os
from ultralytics import YOLO

param_results = []

for cfg in MODELS:
    best = f"runs/detect/{cfg['name']}/weights/best.pt"
    model_name = cfg['weights'].replace('.pt', '')
    model = YOLO(best)

    n_params = sum(p.numel() for p in model.model.parameters())
    file_mb  = os.path.getsize(best) / 1e6

    param_results.append({
        "Model"        : model_name,
        "Parameters (M)": round(n_params / 1e6, 2),
        "Weight file (MB)": round(file_mb, 1),
    })
    print(f"{model_name}: {n_params/1e6:.2f}M params | {file_mb:.1f} MB")

df_params = pd.DataFrame(param_results)
df_params.to_csv("thesis_model_size.csv", index=False)

yolov8n: 3.01M params | 6.2 MB
yolov8s: 11.14M params | 22.5 MB
yolov8m: 25.86M params | 52.0 MB


## 10 — Combined thesis comparison table

In [16]:
df_final = df[["Model","Precision","Recall","mAP@0.5","mAP@0.5:0.95","Train time (min)"]]\
    .merge(df_params[["Model","Parameters (M)","Weight file (MB)"]], on="Model")

print("=== FINAL THESIS TABLE ===")
print(df_final.to_string(index=False))
df_final.to_csv("thesis_final_table.csv", index=False)
print("\nSaved to thesis_final_table.csv")

=== FINAL THESIS TABLE ===
  Model  Precision  Recall  mAP@0.5  mAP@0.5:0.95  Train time (min)  Parameters (M)  Weight file (MB)
yolov8n     0.6273  0.5225   0.5015        0.1994              12.8            3.01               6.2
yolov8s     0.7399  0.6224   0.5985        0.2181              43.6           11.14              22.5
yolov8m     0.6560  0.5735   0.5532        0.2272              57.4           25.86              52.0

Saved to thesis_final_table.csv


## 12 — Training curve plots (loss & mAP over epochs)

In [17]:
import pandas as pd
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle("Training Curves — YOLOv8n / s / m (Identical Hyperparameters)", fontsize=13)

metrics_to_plot = [
    ("train/box_loss",  "Box Loss (train)"),
    ("train/cls_loss",  "Class Loss (train)"),
    ("train/dfl_loss",  "DFL Loss (train)"),
    ("metrics/mAP50(B)",    "mAP@0.5 (val)"),
    ("metrics/mAP50-95(B)", "mAP@0.5:0.95 (val)"),
    ("val/box_loss",    "Box Loss (val)"),
]
colors = {"yolov8n_thesis": "tab:blue", "yolov8s_thesis": "tab:orange", "yolov8m_thesis": "tab:green"}
labels = {"yolov8n_thesis": "YOLOv8n", "yolov8s_thesis": "YOLOv8s", "yolov8m_thesis": "YOLOv8m"}

for ax, (col, title) in zip(axes.flat, metrics_to_plot):
    for cfg in MODELS:
        csv_path = f"runs/detect/{cfg['name']}/results.csv"
        results_df = pd.read_csv(csv_path)
        results_df.columns = results_df.columns.str.strip()
        if col in results_df.columns:
            ax.plot(results_df["epoch"], results_df[col],
                    label=labels[cfg['name']], color=colors[cfg['name']])
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("thesis_training_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved to thesis_training_curves.png")

<Figure size 1600x800 with 6 Axes>

Saved to thesis_training_curves.png
